<a href="https://colab.research.google.com/github/Bhardwajsimran/mental-health/blob/main/Iris_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==================================================
# IMPORTS
# ==================================================
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

# ==================================================
# LOAD DATASET
# ==================================================
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)

df["species"] = pd.Series(iris.target).map({
    0: "setosa",
    1: "versicolor",
    2: "virginica"
})

# ==================================================
# FEATURE ENGINEERING (SAFE)
# ==================================================
df["sepal_ratio"] = df["sepal length (cm)"] / (df["sepal width (cm)"] + 1e-6)
df["petal_ratio"] = df["petal length (cm)"] / (df["petal width (cm)"] + 1e-6)

df["sepal_area"] = df["sepal length (cm)"] * df["sepal width (cm)"]
df["petal_area"] = df["petal length (cm)"] * df["petal width (cm)"]

# ==================================================
# FEATURES & TARGET
# ==================================================
X = df.drop("species", axis=1)
y = df["species"]

# ==================================================
# TRAIN TEST SPLIT
# ==================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==================================================
# SCALING
# ==================================================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==================================================
# MODELS
# ==================================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=300),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# ==================================================
# TRAIN + EVALUATE
# ==================================================
results = {}

for name, model in models.items():

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print("\n==============================")
    print(name)
    print("==============================")
    print("Accuracy:", round(acc, 4))
    print(classification_report(y_test, y_pred))

# ==================================================
# CROSS VALIDATION
# ==================================================
print("\nCross Validation Scores:")
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5)
    print(f"{name}: {scores.mean():.4f}")

# ==================================================
# BEST MODEL
# ==================================================
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

print("\n==============================")
print("BEST MODEL:", best_model_name)
print("Accuracy:", results[best_model_name])

# ==================================================
# SAVE MODEL
# ==================================================
joblib.dump(best_model, "iris_model.pkl")
joblib.dump(scaler, "iris_scaler.pkl")

print("\nModel Saved Successfully!")

# ==================================================
# SAMPLE PREDICTION (TEST)
# ==================================================
sl, sw, pl, pw = 5.1, 3.5, 1.4, 0.2

new_data = pd.DataFrame([[
    sl,
    sw,
    pl,
    pw,
    sl / (sw + 1e-6),
    pl / (pw + 1e-6),
    sl * sw,
    pl * pw
]], columns=X.columns)

prediction = best_model.predict(scaler.transform(new_data))

print("\nPredicted Species:", prediction[0])


Logistic Regression
Accuracy: 0.9667
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30


KNN
Accuracy: 0.9333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginica       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30


Decision Tree
Accuracy: 0.9
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.89      0.80      0